# 01 · Explore the lake (Bronze, Silver, rejected)

Follow one batch from raw bytes to typed, validated Parquet. All data is synthetic.

In [ ]:
# Run from the repository root (`make notebooks` does). Requires `make demo` first.
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import pandas as pd
import duckdb
pd.set_option("display.width", 140)

## Bronze: the raw payload is kept byte-for-byte, with a manifest

In [ ]:
import json
lake = Path("data/lake")
marker = json.loads((lake / "bronze/_batches/batch_id=b000-historical.json").read_text())
pd.DataFrame(marker["objects"])[["entity", "source_system", "fmt", "record_count", "sha256"]]

In [ ]:
first_object = next(o for o in marker["objects"] if o["entity"] == "transactions")
print((lake / first_object["data_path"]).read_text().splitlines()[0])

## Silver: typed, validated, partitioned by transaction date

In [ ]:
silver = pd.read_parquet(lake / "silver/transactions")  # hive partitions become a column
silver.dtypes

In [ ]:
# Money is integer minor units (Int64), never float.
silver[["transaction_id", "currency_code", "amount_minor", "rate_micros", "amount_mxn_minor"]].head()

## Rejected records keep their original payload and every failed rule

In [ ]:
rejected = pd.read_parquet(lake / "rejected/transactions")
rejected["reasons"].str.split("|").explode().value_counts()

In [ ]:
rejected.loc[rejected["disposition"] == "rejected", ["natural_key", "reasons", "raw_record"]].head(10)

## Exercise

Pick one rejected record, fix its raw JSON by hand and predict which Silver columns it would produce. Then check `src/finflow/transformations/silver.py` to confirm.